# Figure 10: Annual-maximum surges, model versus observations

Modelled against observed annual-maximum surge at GESLA-3 stations (2006–2015) for each experiment, and the differences between experiments.

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import proplot as pplt
from sklearn.metrics import r2_score, mean_squared_error
from matplotlib.colors import Normalize
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde, probplot
from matplotlib.ticker import FormatStrFormatter

In [ ]:
data_dir =  DATA_DIR + "data/4-Processed_for_Visualization/2_global_statistics/OBS_points/Aligned_amax_dates/"

stations_to_drop = "Bundaberg_Burnett_Heads", "Crescent_City_CA", "Barmouth", "Kings_Point", 

gtsm_cc = xr.open_dataset(data_dir + "1-CC_Aligned_to_annualmax_gesla3.nc")
#gtsm_cc = gtsm_cc.where(~gtsm_cc["station_name"].isin(stations_to_drop), drop=True)
gtsm_cc = gtsm_cc.sortby("station_name")
gtsm_cv = xr.open_dataset(data_dir + "2-CV_Aligned_to_annualmax_gesla3.nc")
#gtsm_cv = gtsm_cv.where(~gtsm_cv["station_name"].isin(stations_to_drop), drop=True)
gtsm_cv = gtsm_cv.sortby("station_name")
gtsm_ntr = xr.open_dataset(data_dir + "3-UV10N_Aligned_to_annualmax_gesla3.nc")
#gtsm_ntr = gtsm_ntr.where(~gtsm_ntr["station_name"].isin(stations_to_drop), drop=True)
gtsm_ntr = gtsm_ntr.sortby("station_name")
gtsm_rho = xr.open_dataset(data_dir + "4-UV10RHO_Aligned_to_annualmax_gesla3.nc")
#gtsm_rho = gtsm_rho.where(~gtsm_rho["station_name"].isin(stations_to_drop), drop=True)
gtsm_rho = gtsm_rho.sortby("station_name")
gesla3 = xr.open_dataset(data_dir + "0-GESLA3_annualmax.nc")
#gesla3 = gesla3.where(~gesla3["station_name"].isin(stations_to_drop), drop=True)
gesla3 = gesla3.sortby("station_name")

In [ ]:
# Create a DataFrame from the flattened max_surge values
df_max_surge = pd.DataFrame({
    "GESLA3": gesla3["max_surge"].values.flatten(),
    "GTSM_CC": gtsm_cc["max_surge"].values.flatten(),
    "GTSM_CV": gtsm_cv["max_surge"].values.flatten(),
    "GTSM_NTR": gtsm_ntr["max_surge"].values.flatten(),
    "GTSM_RHO": gtsm_rho["max_surge"].values.flatten()
})

# Threshold for absolute differences
threshold = 2

# Filter rows where the absolute difference is greater than the threshold for any model
filtered_df = df_max_surge[
    (np.abs(df_max_surge["GTSM_CC"] - df_max_surge["GESLA3"]) <= threshold) &
    (np.abs(df_max_surge["GTSM_CV"] - df_max_surge["GESLA3"]) <= threshold) &
    (np.abs(df_max_surge["GTSM_NTR"] - df_max_surge["GESLA3"]) <= threshold) &
    (np.abs(df_max_surge["GTSM_RHO"] - df_max_surge["GESLA3"]) <= threshold)
]

# Extract filtered values for plotting
g3_vals = filtered_df["GESLA3"].values
cc_vals = filtered_df["GTSM_CC"].values
cv_vals = filtered_df["GTSM_CV"].values
ntr_vals = filtered_df["GTSM_NTR"].values
rho_vals = filtered_df["GTSM_RHO"].values

# Calculate differences for filtered data
ntr_rho_vals = ntr_vals - rho_vals
cv_ntr_vals = cv_vals - ntr_vals
cc_cv_vals = cc_vals - cv_vals

# Create a figure with 2 rows and 4 columns of subplots
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(22, 10), 
                        sharex=False, sharey=False, constrained_layout=True)

# Define alphabetic labels for the subplots in the a b c style
abc_labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g']

# Row 1: First plot
titles_and_values_row1 = [
    ('$GTSM_{IFS}$ vs OBS', g3_vals, rho_vals, 'Model surge height (m)'),
    ('$GTSM_{CAD}$ vs OBS', g3_vals, ntr_vals, 'Model surge height (m)'),
    ('$GTSM_{NST}$ vs OBS', g3_vals, cv_vals, 'Model surge height (m)'),
    ('$GTSM_{FIX}$ vs OBS', g3_vals, cc_vals, 'Model surge height (m)')
]

for i, (title, g3_data, model_data, ylabel) in enumerate(titles_and_values_row1):
    xy = np.vstack([g3_data, model_data])
    z = gaussian_kde(xy)(xy) if xy.shape[1] > 1 else np.zeros_like(g3_data)
    
    # Linear regression and scatter plot
    slope, intercept = np.polyfit(g3_data, model_data, 1)
    line = slope * g3_data + intercept
    rmse = mean_squared_error(g3_data, model_data, squared=False)
    correlation = np.corrcoef(g3_data, model_data)[0, 1]

    scatter = axs[0, i].scatter(g3_data, model_data, c=z, cmap='sunset_r', alpha=0.5)
    # Format the intercept correctly with the appropriate sign
    sign = '-' if intercept < 0 else '+'
    intercept_abs = abs(intercept)

    axs[0, i].plot(g3_data, line, 'r-', 
                label=f'Linear fit: $y = {slope:.2f}x {sign} {intercept_abs:.2f}$\nRMSE = {rmse:.2f}, Corr = {correlation:.2f}')
    # Add perfect fit line
    max_val = max(g3_data.max(), model_data.max()) + 0.2
    axs[0, i].plot([0, max_val], [0, max_val], 'k--', label='Perfect fit')

    # Add subplot label
    axs[0, i].text(0.01, 1.065, f"${abc_labels[i]}$", transform=axs[0, i].transAxes, 
                   fontsize=16, fontweight='bold', va='top', ha='left')

    # Set titles and labels
    axs[0, i].set_title(title, loc='left', fontsize=15, fontweight='normal', x=0.08)
    axs[0, i].set_xlabel('Observed surge height (m)', fontsize=13.5)
    axs[0, i].set_ylabel(ylabel, fontsize=14)
    axs[0, i].tick_params(axis='both', which='major', labelsize=12)
    axs[0, i].legend(fontsize=13)

    # Add individual colorbars
    cbar = fig.colorbar(scatter, ax=axs[0, i], orientation='vertical', #label='Density',
                        pad=0.02)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label('Density', fontsize=12)
    
# Row 2: Second plot (skipping the first column)
titles_and_values_row2 = [
    ('', '', '', ''),  # Placeholder for the empty plot
    ('$GTSM_{CAD}$—$GTSM_{IFS}$ vs OBS', 
     g3_vals, ntr_rho_vals, 'Model surge height difference (m)'),
    ('$GTSM_{NST}$—$GTSM_{CAD}$ vs OBS', 
     g3_vals, cv_ntr_vals, 'Model surge height difference (m)'),
    ('$GTSM_{FIX}$—$GTSM_{NST}$ vs OBS', 
     g3_vals, cc_cv_vals, 'Model surge height difference (m)')
]

for i, (title, g3_data, diff_data, ylabel) in enumerate(titles_and_values_row2):
    if title == '':
        axs[1, i].axis('off')
        continue
    xy = np.vstack([g3_data, diff_data])
    z = gaussian_kde(xy)(xy) if xy.shape[1] > 1 else np.zeros_like(g3_data)
    
    # Linear regression and scatter plot
    slope, intercept = np.polyfit(g3_data, diff_data, 1)
    line = slope * g3_data + intercept
    rmse = mean_squared_error(g3_data, diff_data, squared=False)
    correlation = np.corrcoef(g3_data, diff_data)[0, 1]

    scatter = axs[1, i].scatter(g3_data, diff_data, c=z, cmap='viridis', alpha=0.5)
    # Format the intercept correctly with the appropriate sign
    sign = '-' if intercept < 0 else '+'
    intercept_abs = abs(intercept)

    axs[1, i].plot(g3_data, line, 'r-', 
                label=f'Linear fit: $y = {slope:.2f}x {sign} {intercept_abs:.2f}$\nRMSE = {rmse:.2f}, Corr = {correlation:.2f}')
        
    max_val = max(g3_data.max(), diff_data.max()) + 0.2

    # Add subplot label
    axs[1, i].text(0.01, 1.065, f"${abc_labels[i + 3]}$", transform=axs[1, i].transAxes, 
                   fontsize=16, fontweight='bold', va='top', ha='left')

    # Set titles and labels
    axs[1, i].set_title(title, loc='left', fontsize=15, fontweight='normal', x=0.06)
    axs[1, i].set_xlabel('Observed surge height (m)', fontsize=13.5)
    axs[1, i].set_ylabel(ylabel, fontsize=14)
    axs[1, i].tick_params(axis='both', which='major', labelsize=12)
    axs[1, i].legend(fontsize=11.6)
    # Add individual colorbars
    cbar = fig.colorbar(scatter, ax=axs[1, i], orientation='vertical', #label='Density', 
                        pad=0.02)
    cbar.ax.tick_params(labelsize=12)     
    cbar.set_label('Density', fontsize=12)
    formatter = FormatStrFormatter('%.2f')
    axs[1, 3].yaxis.set_major_formatter(formatter)
    
# Save the plot
# Save the plot as PDF and PNG, cropping the bounding box
# Define the target directory
save_dir = 'final_plots'
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save the plot as PDF and PNG in the specified directory
fig.savefig(os.path.join(save_dir, f'global_amax_modelvsobs_horizontal_2.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(save_dir, f'global_amax_modelvsobs_horizontal_2.png'), dpi=600, bbox_inches='tight')

# Adjust the layout to minimize blank spaces
plt.tight_layout()  # Adjust layout
plt.show()